In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
load_dotenv()

llm = ChatAnthropic(model="claude-haiku-5-5", max_retries=1, timeout=20)

In [11]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [12]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).text

    return {'joke': response}

In [13]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).text

    return {'explanation': response}

In [14]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor?\n\nIt was feeling a little *crusty*.',
 'explanation': 'The joke relies on a **pun** on the word "crusty."\n\n- **Literal meaning:** A pizza has a crust, so "crusty" describes it accurately.\n- **Figurative meaning:** "Crusty" can also mean grumpy, irritable, or cranky, as in "a crusty old man."\n\nThe pizza goes to the doctor because it\'s "feeling unwell," which is the reason people visit doctors. The pizza then says it\'s feeling "crusty," which sounds like a symptom a doctor would treat. The joke sets up the expectation that the pizza is sick, then delivers a word that fits the pizza\'s own identity while also sounding like a mood or health complaint.\n\nThe italics on *crusty* emphasize the pun, signaling where the punchline lands.'}

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nIt was feeling a little *crusty*.', 'explanation': 'The joke relies on a **pun** on the word "crusty."\n\n- **Literal meaning:** A pizza has a crust, so "crusty" describes it accurately.\n- **Figurative meaning:** "Crusty" can also mean grumpy, irritable, or cranky, as in "a crusty old man."\n\nThe pizza goes to the doctor because it\'s "feeling unwell," which is the reason people visit doctors. The pizza then says it\'s feeling "crusty," which sounds like a symptom a doctor would treat. The joke sets up the expectation that the pizza is sick, then delivers a word that fits the pizza\'s own identity while also sounding like a mood or health complaint.\n\nThe italics on *crusty* emphasize the pun, signaling where the punchline lands.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cda71-a0ac-6454-8002-c2b93ea928c1'}}, metadata={'source': 'loop', 'step'

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nIt was feeling a little *crusty*.', 'explanation': 'The joke relies on a **pun** on the word "crusty."\n\n- **Literal meaning:** A pizza has a crust, so "crusty" describes it accurately.\n- **Figurative meaning:** "Crusty" can also mean grumpy, irritable, or cranky, as in "a crusty old man."\n\nThe pizza goes to the doctor because it\'s "feeling unwell," which is the reason people visit doctors. The pizza then says it\'s feeling "crusty," which sounds like a symptom a doctor would treat. The joke sets up the expectation that the pizza is sick, then delivers a word that fits the pizza\'s own identity while also sounding like a mood or health complaint.\n\nThe italics on *crusty* emphasize the pun, signaling where the punchline lands.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cda71-a0ac-6454-8002-c2b93ea928c1'}}, metadata={'source': 'loop', 'step

In [18]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the pasta go to the doctor?\n\nBecause it was feeling a little *saucy*... and needed to be *al dente* about its health!\n\nHmm, that one's a bit of a stretch. Here's a simpler one:\n\nWhy did the spaghetti refuse to tell secrets?\n\nBecause it was always getting *tangled* up in conversations!",
 'explanation': 'The **spaghetti joke** works through a pun on **"tangled."**\n\n- **Spaghetti** is long, thin, and easily gets knotted into a messy tangle.\n- **"Tangled up in conversations"** is a figure of speech meaning someone gets confused or caught up in complicated talk.\n- The joke implies spaghetti is a poor keeper of secrets because it\'s always getting **tangled** in conversations, so it can\'t keep anything straight, and secrets might slip out.\n\nThe humor comes from applying a literal property of spaghetti (tangling) to a social situation, which makes it sound like the pasta is a gossip who can\'t keep its story straight.'}

In [20]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the doctor?\n\nBecause it was feeling a little *saucy*... and needed to be *al dente* about its health!\n\nHmm, that one's a bit of a stretch. Here's a simpler one:\n\nWhy did the spaghetti refuse to tell secrets?\n\nBecause it was always getting *tangled* up in conversations!", 'explanation': 'The **spaghetti joke** works through a pun on **"tangled."**\n\n- **Spaghetti** is long, thin, and easily gets knotted into a messy tangle.\n- **"Tangled up in conversations"** is a figure of speech meaning someone gets confused or caught up in complicated talk.\n- The joke implies spaghetti is a poor keeper of secrets because it\'s always getting **tangled** in conversations, so it can\'t keep anything straight, and secrets might slip out.\n\nThe humor comes from applying a literal property of spaghetti (tangling) to a social situation, which makes it sound like the pasta is a gossip who can\'t keep its story straight.'}, 

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nIt was feeling a little *crusty*.', 'explanation': 'The joke relies on a **pun** on the word "crusty."\n\n- **Literal meaning:** A pizza has a crust, so "crusty" describes it accurately.\n- **Figurative meaning:** "Crusty" can also mean grumpy, irritable, or cranky, as in "a crusty old man."\n\nThe pizza goes to the doctor because it\'s "feeling unwell," which is the reason people visit doctors. The pizza then says it\'s feeling "crusty," which sounds like a symptom a doctor would treat. The joke sets up the expectation that the pizza is sick, then delivers a word that fits the pizza\'s own identity while also sounding like a mood or health complaint.\n\nThe italics on *crusty* emphasize the pun, signaling where the punchline lands.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cda71-a0ac-6454-8002-c2b93ea928c1'}}, metadata={'source': 'loop', 'step

Time Travel

In [23]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1cda71-7adb-6454-8000-458750a4de56"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1cda71-7adb-6454-8000-458750a4de56'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-21T23:28:16.568840+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cda71-7ad3-6f6e-bfff-5526387b6091'}}, tasks=(PregelTask(id='4f009fd0-421d-77b5-f9c2-f4f7e25a4b22', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the doctor?\n\nIt was feeling a little *crusty*.'}),), interrupts=())

Fault Tolerance

In [24]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [25]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [26]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [27]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))